In [1]:
import yfinance as yf
import pandas as pd
import os

In [2]:
# daily RUB/USD from 2018 to today
fx_ur = yf.download("RUBUSD=X", start="2015-01-01", end="2026-05-04", 
                 interval="1d", auto_adjust=False, multi_level_index=False)

[*********************100%***********************]  1 of 1 completed


In [3]:
fx_ur

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2015-01-01,0.016474,0.016474,0.017145,0.017145,0.017145,0
2015-01-02,0.017145,0.017145,0.017354,0.016671,0.016812,0
2015-01-05,0.017021,0.017021,0.017466,0.016761,0.017058,0
2015-01-06,0.016439,0.016439,0.016569,0.015648,0.016569,0
2015-01-07,0.015833,0.015833,0.016128,0.015728,0.015828,0
...,...,...,...,...,...,...
2026-04-28,0.013357,0.013357,0.013397,0.013223,0.013356,0
2026-04-29,0.013276,0.013276,0.013381,0.013254,0.013277,0
2026-04-30,0.013379,0.013379,0.013382,0.013323,0.013379,0


In [4]:
fx_ur.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2950 entries, 2015-01-01 to 2026-05-04
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  2950 non-null   float64
 1   Close      2950 non-null   float64
 2   High       2950 non-null   float64
 3   Low        2950 non-null   float64
 4   Open       2950 non-null   float64
 5   Volume     2950 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 161.3 KB


In [5]:
fx_ur.isnull().sum()

Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

In [6]:
fx_ur.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

In [7]:
rub_usd = fx_ur[["Close"]].rename(columns={"Close": "rub_usd"})

In [8]:
rub_usd = rub_usd.reset_index() # date is alr in datetime
rub_usd.columns = ["date", "rub_usd"]
rub_usd["date"] = pd.to_datetime(rub_usd["date"])

In [9]:
rub_usd

,date,rub_usd
0,2015-01-01,0.016474
1,2015-01-02,0.017145
2,2015-01-05,0.017021
3,2015-01-06,0.016439
4,2015-01-07,0.015833
...,...,...
2945,2026-04-28,0.013357
2946,2026-04-29,0.013276
2947,2026-04-30,0.013379
2948,2026-05-01,0.013341


In [10]:
rub_usd.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2950 entries, 0 to 2949
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     2950 non-null   datetime64[ns]
 1   rub_usd  2950 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 46.2 KB


In [11]:
rub_usd = rub_usd.set_index("date").sort_index()

In [12]:
weekly = rub_usd.resample("W-FRI").agg(
    rub_usd_close=("rub_usd", "last"),
    rub_usd_mean=("rub_usd", "mean")
)
weekly.index = weekly.index + pd.Timedelta(days=3)
weekly.index.name = "week_monday"

In [13]:
weekly

,rub_usd_close,rub_usd_mean
week_monday,,
2015-01-05,0.017145,0.016810
2015-01-12,0.016598,0.016370
2015-01-19,0.015461,0.015704
2015-01-26,0.015551,0.015403
2015-02-02,0.014549,0.014877
...,...,...
2026-04-13,0.012883,0.012655
2026-04-20,0.013094,0.013160
2026-04-27,0.013180,0.013279


In [14]:
weekly.to_parquet("../data/semi-cleaned/fx_rub_usd.parquet")